# Class 4 — Bringing It Together: Joins, Aggregation & Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MostafaElFar96/QaF_Data_Analytics/blob/main/notebooks/Class_4_Joins_and_Visuals.ipynb)

Module 4 · Sanad Distribution case (fictional data). Run the cells top to bottom. Works in **Google Colab** (nothing to install) and in **VS Code** (Python + Jupyter extension).

## Setup

In [ ]:
# --- Setup: works in Google Colab and in VS Code -------------------------------
import os, sys, subprocess
if os.path.exists("../data/generate_sanad_data.py"):        # opened from the repo's notebooks/ folder
    os.chdir("..")
elif not os.path.exists("data/generate_sanad_data.py"):     # Colab: get the repo first
    if not os.path.exists("QaF_Data_Analytics"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/MostafaElFar96/QaF_Data_Analytics"], check=True)
    os.chdir("QaF_Data_Analytics")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "beautifulsoup4"], check=True)

if not os.path.exists("data/sanad/orders.csv"):            # same seed -> same data for everyone (~15 s)
    subprocess.run([sys.executable, "data/generate_sanad_data.py"], check=True)

import duckdb, pandas as pd
con = duckdb.connect("data/sanad.duckdb")

def run(statement):
    """Execute a statement (CREATE, INSERT...) and finish it."""
    con.execute(statement).fetchall()

def scalar(query):
    """Return the single value of a one-cell query."""
    return con.execute(query).fetchall()[0][0]
TABLES = ["areas", "retailers", "warehouses", "orders", "deliveries", "stock",
          "candidate_sites", "drive_times", "fsq_places_sample"]
existing = {r[0] for r in con.execute("SELECT table_name FROM information_schema.tables").fetchall()}
for t in TABLES:
    if t not in existing:
        run(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('data/sanad/{t}.csv')")

def sql(query):
    """Run SQL and return a pandas DataFrame (DuckDB speaks the same SQL as PostgreSQL here)."""
    return con.execute(query).df()

print("Ready:", ", ".join(f"{t} ({scalar(f'SELECT COUNT(*) FROM {t}'):,})" for t in TABLES[:6]))

## Video 1 — The fan-out trap

In [ ]:
sql("""
-- WRONG: each retailer row repeats once per candidate site
SELECT COUNT(*) AS rows_after_join
FROM retailers r
JOIN drive_times t ON t.area_id = r.area_id
""")  # 4,000 retailers x 7 sites = 28,000

## Video 1 — Bring demand to the common grain (area), join, then aggregate to site

In [ ]:
run("""
CREATE OR REPLACE VIEW v_area_demand AS
SELECT r.area_id,
       SUM(o.value_egp) / 6           AS monthly_value_egp,
       COUNT(DISTINCT r.retailer_id)  AS shops
FROM orders o
JOIN retailers r USING (retailer_id)
WHERE o.order_date >= DATE '2026-05-01'   -- last 6 months
GROUP BY r.area_id
""")
coverage = sql("""
SELECT s.site_id, s.name, s.rent_egp,
       SUM(CASE WHEN t.minutes_10am <= 45 THEN d.monthly_value_egp ELSE 0 END)
         / SUM(d.monthly_value_egp)     AS coverage_45min
FROM candidate_sites s
JOIN drive_times   t USING (site_id)
JOIN v_area_demand d USING (area_id)
JOIN areas         a USING (area_id)
WHERE a.cluster = 'New Cairo'
GROUP BY 1, 2, 3
ORDER BY coverage_45min DESC
""")
coverage.assign(coverage_45min=(coverage.coverage_45min * 100).round().astype(int).astype(str) + "%")

## Video 1 — Why the census is not joined
City Population gives population per official district (qism); those borders don't match Sanad's delivery areas, and 2023 is an estimate. Use it as map context only.

## Video 1 — Extra analysis: untapped shops (Foursquare-style data)
Places have coordinates, so we can (1) drop those already served by Sanad (a Sanad shop within 50 m) and (2) place each one in its delivery area. Here 'area' = nearest area centre — a simple stand-in for a true point-in-polygon join.

In [ ]:
import numpy as np
places = sql("SELECT * FROM fsq_places_sample")
shops = sql("SELECT retailer_id, lat, lon FROM retailers WHERE lat IS NOT NULL")
centres = sql("SELECT area_id, area_name, lat, lon FROM areas WHERE cluster = 'New Cairo' AND area_name <> 'Tagamoa 5'")
def km(lat1, lon1, lat2, lon2):
    return 111.0 * np.hypot(lat1 - lat2, (lon1 - lon2) * np.cos(np.radians(30.05)))
places["nearest_shop_km"] = [km(la, lo, shops.lat.values, shops.lon.values).min() for la, lo in zip(places.latitude, places.longitude)]
places["served"] = places.nearest_shop_km < 0.05
places["area_id"] = [centres.area_id.values[np.argmin(km(la, lo, centres.lat.values, centres.lon.values))]
                     for la, lo in zip(places.latitude, places.longitude)]
unserved = places[~places.served]
print(f"{places.served.sum()} already served, {len(unserved)} untapped shops")
reach = sql("SELECT site_id, area_id, minutes_10am FROM drive_times").query("minutes_10am <= 45")
for s in ["E", "G"]:
    n = unserved.area_id.isin(reach[reach.site_id == s].area_id).sum()
    print(f"Site {s}: {coverage.set_index('site_id').coverage_45min[s]:.0%} coverage, {n} untapped shops within 45 min")

## Video 2 — Which site covers the most? (sorted bar)

In [ ]:
import matplotlib.pyplot as plt
c = coverage.sort_values("coverage_45min")
plt.figure(figsize=(8, 4)); plt.barh(c.site_id, c.coverage_45min * 100, color="#5e7c8a")
for y, v in enumerate(c.coverage_45min * 100): plt.text(v + 1, y, f"{v:.0f}%", va="center")
plt.title("Which site covers the most New Cairo demand within 45 min?"); plt.xlim(0, 105); plt.show()

## Video 2 — Is the coverage worth the rent? (scatter + rent cap)

In [ ]:
plt.figure(figsize=(8, 5))
for _, r in coverage.iterrows():
    over = r.rent_egp > 450000
    plt.scatter(r.rent_egp / 1000, r.coverage_45min * 100, s=120,
                color="#b0b0b0" if over else ("#e85d04" if r.site_id in "EG" else "#14213d"))
    plt.annotate(r.site_id + (" (over cap)" if over else ""), (r.rent_egp / 1000 + 4, r.coverage_45min * 100 + 1))
plt.axvline(450, ls="--", color="black"); plt.text(452, 50, "rent cap")
plt.xlabel("Monthly rent (EGP K)"); plt.ylabel("45-min coverage (%)"); plt.title("Is the coverage worth the rent?"); plt.show()

## Video 2 — Where are the gaps? (map) and is demand growing? (lines)

In [ ]:
SITE = "E"
gap = sql(f"""
SELECT a.area_name, a.lat, a.lon, t.minutes_10am, d.monthly_value_egp
FROM areas a JOIN drive_times t USING (area_id) JOIN v_area_demand d USING (area_id)
WHERE a.cluster = 'New Cairo' AND t.site_id = '{SITE}'
""")
band = pd.cut(gap.minutes_10am, [0, 30, 45, 999], labels=["<= 30 min", "30-45 min", "> 45 min"])
colors = {"<= 30 min": "#2a9d8f", "30-45 min": "#e9c46a", "> 45 min": "#e76f51"}
site = sql(f"SELECT lat, lon FROM candidate_sites WHERE site_id = '{SITE}'").iloc[0]
plt.figure(figsize=(8, 6))
for b, g in gap.groupby(band, observed=True):
    plt.scatter(g.lon, g.lat, s=g.monthly_value_egp / 2e4, color=colors[b], alpha=.7, label=b)
for _, r in gap.iterrows(): plt.annotate(r.area_name, (r.lon, r.lat), fontsize=8, ha="center")
plt.scatter(site.lon, site.lat, marker="*", s=400, color="black", label=f"Site {SITE}")
plt.legend(title="Drive time at 10 am"); plt.title(f"Where are the gaps? (site {SITE})"); plt.show()

In [ ]:
trend = sql("""
SELECT DATE_TRUNC('month', o.order_date) AS month,
       CASE WHEN a.area_name = 'Tagamoa 5' THEN 'Fifth Settlement' ELSE a.area_name END AS area,
       SUM(o.value_egp) / 1e6 AS value_m
FROM orders o JOIN retailers r USING (retailer_id) JOIN areas a USING (area_id)
WHERE a.cluster = 'New Cairo' GROUP BY 1, 2 ORDER BY 1
""").pivot(index="month", columns="area", values="value_m")
trend.plot(figsize=(9, 4), title="Is demand growing where we'd be? (EGP M / month)"); plt.legend(fontsize=7, ncol=2); plt.show()

## Video 3 — Drill-down: site → areas → shops (key accounts flagged)

In [ ]:
SITE = "E"
areas_for_site = sql(f"""
SELECT a.area_id, a.area_name, t.minutes_10am, ROUND(d.monthly_value_egp) AS monthly_value_egp,
       t.minutes_10am <= 45 AS covered
FROM drive_times t JOIN areas a USING (area_id) JOIN v_area_demand d USING (area_id)
WHERE t.site_id = '{SITE}' AND a.cluster = 'New Cairo' ORDER BY covered, monthly_value_egp DESC
""")
areas_for_site

In [ ]:
uncovered_ids = areas_for_site.loc[~areas_for_site.covered, "area_id"].tolist()
key_accounts_far = sql(f"""
SELECT r.retailer_id, a.area_name, r.segment, r.is_top200 AS key_account
FROM retailers r JOIN areas a USING (area_id)
WHERE r.area_id IN ({",".join(map(str, uncovered_ids)) or "NULL"}) AND r.is_top200
""")
print(f"Key accounts still beyond 45 min from site {SITE}: {len(key_accounts_far)}"); key_accounts_far.head(10)

## Video 3 — Filters that change the story: time of day
6 am ≈ free-flow × 1.1 (illustrative).

In [ ]:
def coverage_at(minutes_col):
    return sql(f"""
    SELECT s.site_id, ROUND(100 * SUM(CASE WHEN {minutes_col} <= 45 THEN d.monthly_value_egp ELSE 0 END) / SUM(d.monthly_value_egp)) AS pct
    FROM candidate_sites s JOIN drive_times t USING (site_id) JOIN v_area_demand d USING (area_id) JOIN areas a USING (area_id)
    WHERE a.cluster = 'New Cairo' GROUP BY 1""").set_index("site_id").pct
pd.DataFrame({"10 am": coverage_at("t.minutes_10am"), "6 am": coverage_at("t.minutes_free_flow * 1.1")}).sort_values("10 am", ascending=False)